In [1]:
import torch
import torch.nn as nn
import pandas as pd
from einops import rearrange, repeat
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os

from tab_transformer_pytorch import TabTransformer as BaseTabTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def exists(val):
    return val is not None

def default(val, d):
    return val if exists(val) else d


In [2]:
class TabTransformerwithEmbedding(BaseTabTransformer):
    def forward(self, x_categ, x_cont, return_attn=False, return_embedding=False):
        xs = []

        assert x_categ.shape[-1] == self.num_categories
        if self.num_unique_categories > 0:
            x_categ = x_categ + self.categories_offset
            categ_embed = self.category_embed(x_categ)

            if self.use_shared_categ_embed:
                shared_categ_embed = repeat(self.shared_category_embed, 'n d -> b n d', b=categ_embed.shape[0])
                categ_embed = torch.cat((categ_embed, shared_categ_embed), dim=-1)

            x, attns = self.transformer(categ_embed, return_attn=True)
            flat_categ = rearrange(x, 'b ... -> b (...)')
            xs.append(flat_categ)

        if self.num_continuous > 0:
            if exists(self.continuous_mean_std):
                mean, std = self.continuous_mean_std.unbind(dim=-1)
                x_cont = (x_cont - mean) / std
            normed_cont = self.norm(x_cont)
            xs.append(normed_cont)

        x = torch.cat(xs, dim=-1)
        x = self.embedding_proj(x)

        if return_embedding:
            if return_attn:
                return x, attns
            return x

        logits = self.mlp(x)
        if return_attn:
            return logits, attns
        return logits


In [3]:
# Load the Excel file
imputed_df1 = pd.read_csv("/home/UT_shared/data/train.csv")
imputed_df_train = pd.read_csv("/home/UT_shared/data/train.csv").drop(columns=['year', 'status'])
outcome_train = pd.read_csv('/home/UT_shared/data/train.csv')[['year','status']]
y_train = torch.tensor(imputed_df1['status'].to_numpy(), dtype=torch.long)

imputed_df2 = pd.read_csv("/home/UT_shared/data/val.csv")
imputed_df_val = pd.read_csv("/home/UT_shared/data/val.csv").drop(columns=['year', 'status'])
outcome_val = pd.read_csv('/home/UT_shared/data/val.csv')[['year','status']]
y_val = torch.tensor(imputed_df2['status'].to_numpy(), dtype=torch.long)

imputed_df3 = pd.read_csv("/home/UT_shared/data/test.csv")
imputed_df_test = pd.read_csv("/home/UT_shared/data/test.csv").drop(columns=['year', 'status'])
outcome_test = pd.read_csv('/home/UT_shared/data/test.csv')[['year','status']]
y_test = torch.tensor(imputed_df3['status'].to_numpy(), dtype=torch.long)


In [4]:
# Save original column names first
original_columns = df.columns.tolist()
categ_idx = list(range(2, 38)) + list(range(42, 53)) + list(range(72, 108))
cont_idx = [1] + list(range(38, 42)) + list(range(53, 72)) 

# Convert column index positions to column names
categ_cols = imputed_df_train.columns[categ_idx]
cont_cols = imputed_df_train.columns[cont_idx]


NameError: name 'df' is not defined

In [5]:
# Extract features
x_categ_train = torch.tensor(imputed_df_train[categ_cols].to_numpy())
x_cont_train = torch.tensor(imputed_df_train[cont_cols].to_numpy(), dtype=torch.float32)

x_categ_val = torch.tensor(imputed_df_val[categ_cols].to_numpy())
x_cont_val = torch.tensor(imputed_df_val[cont_cols].to_numpy(), dtype=torch.float32)

x_categ_test = torch.tensor(imputed_df_test[categ_cols].to_numpy())
x_cont_test = torch.tensor(imputed_df_test[cont_cols].to_numpy(), dtype=torch.float32)

print(x_categ_test.shape)
print(x_cont_test.shape)

NameError: name 'categ_cols' is not defined

In [ ]:
model = TabTransformerwithEmbedding(
    categories = (2,)*83,      # tuple containing the number of unique values within each category
    num_continuous = 24,                # number of continuous values
    dim = 32,                           # dimension, paper set at 32
    dim_out = 3,                        # binary prediction, but could be anything
    depth = 6,                          # depth, paper recommended 6
    heads = 8,                          # heads, paper recommends 8
    attn_dropout = 0.1,                 # post-attention dropout
    ff_dropout = 0.1,                   # feed forward dropout
    mlp_hidden_mults = (4, 2),          # relative multiples of each hidden dimension of the last mlp to logits
    mlp_act = nn.ReLU()                 # activation for final mlp, defaults to relu, but could be anything else (selu etc)
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

x_categ_train, x_cont_train, y_train = x_categ_train.to(device), x_cont_train.to(device), y_train.to(device)
x_categ_val, x_cont_val, y_val = x_categ_val.to(device), x_cont_val.to(device), y_val.to(device)
# --- Create dataset and dataloader ---
train_dataset = TensorDataset(x_categ_train, x_cont_train, y_train)
val_dataset = TensorDataset(x_categ_val, x_cont_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

best_val_loss = float('inf')
save_path = "/data/lab_ph/yushi/ehr/results/best_tabtransformer_survival_model_110_500_1.pth"


In [ ]:
for epoch in range(30):  # change as needed
    model.train()
    train_loss, correct_train, total_train = 0, 0, 0
    for batch_categ, batch_cont, batch_y in train_loader:
        batch_categ, batch_cont, batch_y = batch_categ.to(device), batch_cont.to(device), batch_y.to(device)
        optimizer.zero_grad()
        preds = model(batch_categ, batch_cont)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_y.size(0)
        correct_train += (preds.argmax(dim=1) == batch_y).sum().item()
        total_train += batch_y.size(0)

    train_acc = correct_train / total_train

    # Validation
    model.eval()
    val_loss, correct_val, total_val = 0, 0, 0
    with torch.no_grad():
        for val_categ, val_cont, val_y in val_loader:
            val_categ, val_cont, val_y = val_categ.to(device), val_cont.to(device), val_y.to(device)
            preds = model(val_categ, val_cont)
            loss = criterion(preds, val_y)

            val_loss += loss.item() * val_y.size(0)
            correct_val += (preds.argmax(dim=1) == val_y).sum().item()
            total_val += val_y.size(0)

    val_acc = correct_val / total_val

    print(f"Epoch {epoch:03}: Train Loss={train_loss/total_train:.4f}, Acc={train_acc:.4f} | "
          f"Val Loss={val_loss/total_val:.4f}, Acc={val_acc:.4f}")


    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print(f"Saved best model at epoch {epoch} with val loss {val_loss/total_val:.4f}")
